# Notebook 07 — Embeddings & Vector Store

This notebook generates text documents from the clean structured IDSP data,
embeds them using the **Gemini Embedding API**, and stores them in **ChromaDB**
for semantic retrieval.

**Why not embed raw PDF text?**  
Structured data produces cleaner, more queryable embeddings. Each document
contains district + disease + counts + localities in natural language,
with metadata for filtered retrieval.

**Prerequisites:**
```
pip install chromadb google-generativeai python-dotenv
```
Set `GEMINI_API_KEY` in `.env`.

## 1. Setup & API Connection

In [18]:
import os
import sqlite3
import pandas as pd
from dotenv import load_dotenv
from google import genai
import chromadb

load_dotenv(os.path.join("..", ".env"))

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in .env")

gemini_client = genai.Client(api_key=GEMINI_API_KEY)
print("Gemini API client created.")

DB_PATH = "data/idsp_kerala.db"
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
print(f"Connected to {DB_PATH}")

Gemini API client created.
Connected to data/idsp_kerala.db


## 2. Generate Text Documents from Structured Data

We create three types of documents for embedding:
1. **District-disease summaries** — one per district per disease with case counts
2. **Locality documents** — disease + sub-district location details
3. **Death notes** — individual death records with demographics

In [19]:
documents = []
metadatas = []
ids = []

# --- Type 1: District-disease summaries ---
df = pd.read_sql_query("""
    SELECT r.report_date, o.district_name, o.district_code, o.disease,
           o.metric, o.subtype, o.value
    FROM observations o
    JOIN reports r ON o.report_id = r.report_id
    WHERE o.value > 0
    ORDER BY o.district_name, o.disease
""", conn)

for (date, district, code, disease), group in df.groupby(
    ["report_date", "district_name", "district_code", "disease"]
):
    lines = [f"Report date: {date}",
             f"District: {district} ({code})",
             f"Disease: {disease}"]
    for _, row in group.iterrows():
        metric_label = row["metric"]
        if pd.notna(row["subtype"]) and row["subtype"]:
            metric_label += f" ({row['subtype']})"
        lines.append(f"  {metric_label}: {int(row['value'])}")
    lines.append(f"Source: Kerala IDSP Daily Report, page 2.")

    doc_id = f"obs_{date}_{code}_{disease.replace(' ', '_')}"
    documents.append("\n".join(lines))
    metadatas.append({
        "report_date": date,
        "district": district,
        "district_code": code,
        "disease": disease,
        "doc_type": "district_disease_summary",
        "source_page": 2
    })
    ids.append(doc_id)

print(f"District-disease summaries: {len(ids)} documents")

# --- Type 2: Locality documents ---
loc_df = pd.read_sql_query("""
    SELECT r.report_date, lr.district_name, lr.district_code, lr.disease,
           lr.district_reported_count, lr.locality_text
    FROM locality_reports lr
    JOIN reports r ON lr.report_id = r.report_id
""", conn)

loc_count = 0
for _, row in loc_df.iterrows():
    count_str = f"{int(row['district_reported_count'])} confirmed cases" if pd.notna(row['district_reported_count']) else "cases reported"
    text = (
        f"Report date: {row['report_date']}\n"
        f"District: {row['district_name']}\n"
        f"Disease: {row['disease']}\n"
        f"Confirmed cases in district: {count_str}\n"
        f"Reported localities: {row['locality_text']}\n"
        f"Source: Kerala IDSP Daily Report, page 2 (locality section)."
    )
    doc_id = f"loc_{row['report_date']}_{row['district_code']}_{row['disease'].replace(' ', '_')}"
    documents.append(text)
    metadatas.append({
        "report_date": row["report_date"],
        "district": row["district_name"],
        "district_code": row["district_code"],
        "disease": row["disease"],
        "doc_type": "locality_report",
        "source_page": 2
    })
    ids.append(doc_id)
    loc_count += 1

print(f"Locality documents: {loc_count}")

# --- Type 3: Death notes ---
death_df = pd.read_sql_query("""
    SELECT r.report_date, dn.district_name, dn.district_code, dn.disease,
           dn.age, dn.sex, dn.locality, dn.death_date, dn.raw_text
    FROM death_notes dn
    JOIN reports r ON dn.report_id = r.report_id
""", conn)

death_count = 0
for idx, row in death_df.iterrows():
    text = (
        f"Report date: {row['report_date']}\n"
        f"District: {row['district_name']}\n"
        f"Disease: {row['disease']}\n"
        f"Death: {row['age']}-year-old {row['sex']} from {row['locality']}\n"
        f"Date of death: {row['death_date']}\n"
        f"Source: Kerala IDSP Daily Report, page 3 (death notes)."
    )
    doc_id = f"death_{row['report_date']}_{row['district_code']}_{idx}"
    documents.append(text)
    metadatas.append({
        "report_date": row["report_date"],
        "district": row["district_name"],
        "district_code": row.get("district_code", ""),
        "disease": row["disease"],
        "doc_type": "death_note",
        "source_page": 3
    })
    ids.append(doc_id)
    death_count += 1

print(f"Death note documents: {death_count}")
print(f"\nTotal documents to embed: {len(documents)}")

District-disease summaries: 86 documents
Locality documents: 24
Death note documents: 6

Total documents to embed: 116


### Preview a few documents

In [20]:
for i in [0, len(documents)//2, -1]:
    print(f"--- Document [{i}] (type: {metadatas[i]['doc_type']}) ---")
    print(documents[i])
    print(f"Metadata: {metadatas[i]}")
    print()

--- Document [0] (type: district_disease_summary) ---
Report date: 2026-09-01
District: Alappuzha (ALP)
Disease: Acute Diarrhoeal Disease
  confirmed: 81
Source: Kerala IDSP Daily Report, page 2.
Metadata: {'report_date': '2026-09-01', 'district': 'Alappuzha', 'district_code': 'ALP', 'disease': 'Acute Diarrhoeal Disease', 'doc_type': 'district_disease_summary', 'source_page': 2}

--- Document [58] (type: district_disease_summary) ---
Report date: 2026-09-01
District: Palakkad (PKD)
Disease: Fever
  outpatient: 1114
  inpatient: 7
Source: Kerala IDSP Daily Report, page 2.
Metadata: {'report_date': '2026-09-01', 'district': 'Palakkad', 'district_code': 'PKD', 'disease': 'Fever', 'doc_type': 'district_disease_summary', 'source_page': 2}

--- Document [-1] (type: death_note) ---
Report date: 2026-09-01
District: Malappuram
Disease: Amoebic Meningoencephalitis
Death: 38-year-old male from Marakkara
Date of death: 2026-08-31
Source: Kerala IDSP Daily Report, page 3 (death notes).
Metadata: {

## 3. Generate Embeddings with Gemini API

Using `models/embedding-001`. The API supports batch embedding,
but we'll send in chunks to avoid rate limits.

In [21]:
import time

EMBEDDING_MODEL = "gemini-embedding-001"
BATCH_SIZE = 20

all_embeddings = []

for i in range(0, len(documents), BATCH_SIZE):
    batch = documents[i:i + BATCH_SIZE]
    result = gemini_client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=batch,
        config={"task_type": "RETRIEVAL_DOCUMENT"}
    )
    all_embeddings.extend([e.values for e in result.embeddings])
    print(f"Embedded batch {i // BATCH_SIZE + 1}/{(len(documents) - 1) // BATCH_SIZE + 1} ({len(batch)} docs)")
    time.sleep(1)

print(f"\nTotal embeddings generated: {len(all_embeddings)}")
print(f"Embedding dimension: {len(all_embeddings[0])}")

Embedded batch 1/6 (20 docs)
Embedded batch 2/6 (20 docs)
Embedded batch 3/6 (20 docs)
Embedded batch 4/6 (20 docs)
Embedded batch 5/6 (20 docs)


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 6.349693777s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-embedding-1.0', 'locat

## 4. Store in ChromaDB

In [ ]:
CHROMA_PATH = os.path.join("..", "data", "chroma_db")
os.makedirs(CHROMA_PATH, exist_ok=True)

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# Delete existing collection if re-running
try:
    chroma_client.delete_collection("idsp_kerala")
    print("Deleted existing collection.")
except ValueError:
    pass

collection = chroma_client.create_collection(
    name="idsp_kerala",
    metadata={"description": "IDSP Kerala disease surveillance embeddings"}
)

# Add in batches (ChromaDB limit)
CHROMA_BATCH = 100
for i in range(0, len(documents), CHROMA_BATCH):
    end = min(i + CHROMA_BATCH, len(documents))
    collection.add(
        ids=ids[i:end],
        documents=documents[i:end],
        embeddings=all_embeddings[i:end],
        metadatas=metadatas[i:end]
    )

print(f"Stored {collection.count()} documents in ChromaDB at {CHROMA_PATH}")

## 5. Test Semantic Retrieval

Query the vector store with natural language questions.

In [ ]:
def query_vector_store(question, n_results=5, where=None):
    """Embed the question and retrieve similar documents from ChromaDB."""
    q_result = gemini_client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=question,
        config={"task_type": "RETRIEVAL_QUERY"}
    )
    q_embedding = q_result.embeddings[0].values

    kwargs = {
        "query_embeddings": [q_embedding],
        "n_results": n_results,
    }
    if where:
        kwargs["where"] = where

    return collection.query(**kwargs)


print("Vector search function ready.")

In [ ]:
test_queries = [
    "What is the dengue situation in Thiruvananthapuram?",
    "Where was leptospirosis reported?",
    "Any recent deaths due to dengue?",
    "Which areas in Ernakulam have disease outbreaks?",
    "Tell me about H1N1 deaths",
]

for q in test_queries:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    results = query_vector_store(q, n_results=3)
    for i, (doc, meta, dist) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    )):
        print(f"\n  Result {i+1} (distance: {dist:.4f}) [{meta['doc_type']}]")
        print(f"  {doc[:200]}..." if len(doc) > 200 else f"  {doc}")

### Test with metadata filtering

In [ ]:
# Filter: only locality documents in Kannur
results = query_vector_store(
    "dengue cases",
    n_results=3,
    where={"$and": [{"district": "Kannur"}, {"doc_type": "locality_report"}]}
)

print("Filtered query: dengue cases in Kannur (locality docs only)")
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(f"\n  [{meta['doc_type']}] {meta['district']} - {meta['disease']}")
    print(f"  {doc[:300]}")

## 6. Export Vector Search Module

In [ ]:
module_code = '''"""IDSP Kerala — Vector Search Module"""

import os
import chromadb
from google import genai
from dotenv import load_dotenv
from typing import Optional


EMBEDDING_MODEL = "gemini-embedding-001"


class VectorSearch:
    def __init__(self, chroma_path: str, env_path: str = ".env"):
        load_dotenv(env_path)
        api_key = os.getenv("GEMINI_API_KEY")
        if not api_key:
            raise ValueError("GEMINI_API_KEY not found in .env")
        self.gemini_client = genai.Client(api_key=api_key)

        chroma_client = chromadb.PersistentClient(path=chroma_path)
        self.collection = chroma_client.get_collection("idsp_kerala")

    def search(self, question: str, n_results: int = 5,
               where: Optional[dict] = None) -> dict:
        q_result = self.gemini_client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=question,
            config={"task_type": "RETRIEVAL_QUERY"}
        )
        q_embedding = q_result.embeddings[0].values

        kwargs = {
            "query_embeddings": [q_embedding],
            "n_results": n_results,
        }
        if where:
            kwargs["where"] = where

        return self.collection.query(**kwargs)

    def get_context(self, question: str, n_results: int = 5,
                    where: Optional[dict] = None) -> str:
        results = self.search(question, n_results, where)
        if not results["documents"][0]:
            return "No relevant documents found."
        context_parts = []
        for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
            context_parts.append(f"[{meta['doc_type']}] {doc}")
        return "\\n\\n---\\n\\n".join(context_parts)
'''

module_path = os.path.join("..", "vector_search.py")
with open(module_path, "w") as f:
    f.write(module_code)
print(f"Saved vector search module to {module_path}")

In [ ]:
conn.close()
print("Database connection closed.")
print(f"ChromaDB stored at: {os.path.abspath(CHROMA_PATH)}")
print(f"Total documents in vector store: {collection.count()}")